# AAT6153 Chapter 1 — Hands-on 1
## Building a Neural Network with TensorFlow / Keras
**Slides:** Neural Network Basics (Neuron, Network of Neurons, Training Neural Networks) and Hands-on 1 (slides 6–23)

In this notebook you will:
1. Compute the output of a **single artificial neuron** by hand and plot common **activation functions** (slides 6–7).
2. Build, train and evaluate a **21-12-8-1** neural network that predicts diabetes from health indicators (slides 20–23).
3. Experiment with **topology, learning rate, batch size and early stopping** (slides 12–18).

*Runtime: CPU is fine for this notebook.*

---
# Part A — The artificial neuron (slides 6–7)
A neuron computes a **weighted sum** of its inputs plus a **bias**, then passes it through an **activation function**:

$$y = f\left(\sum_{i} w_i x_i + b\right)$$

A neuron with three inputs needs **four** weights: one per input and one for the bias.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

x = np.array([0.5, -1.2, 3.0])     # three inputs
w = np.array([0.4, 0.1, -0.2])     # one weight per input
b = 0.3                            # bias (the weight on a constant input of 1.0)

z = np.dot(w, x) + b               # weighted sum
print(f"Weighted sum z = {z:.3f}")

sigmoid = lambda z: 1 / (1 + np.exp(-z))
relu = lambda z: np.maximum(0, z)
print(f"Sigmoid output = {sigmoid(z):.3f}")
print(f"ReLU output    = {relu(z):.3f}")

In [ ]:
# Plot common activation functions
z = np.linspace(-5, 5, 200)
acts = {
    "Sigmoid": 1 / (1 + np.exp(-z)),
    "Tanh": np.tanh(z),
    "ReLU": np.maximum(0, z),
    "Leaky ReLU (α=0.1)": np.where(z > 0, z, 0.1 * z),
}
fig, axes = plt.subplots(1, 4, figsize=(16, 3.5))
for ax, (name, y) in zip(axes, acts.items()):
    ax.plot(z, y, lw=2)
    ax.axhline(0, color="grey", lw=0.5); ax.axvline(0, color="grey", lw=0.5)
    ax.set_title(name); ax.set_xlabel("z")
plt.tight_layout(); plt.show()

In [ ]:
# Softmax turns a vector of scores into class probabilities (slide 11)
scores = np.array([2.0, 1.0, 0.1])            # raw outputs for Class 1, 2, 3
probs = np.exp(scores) / np.exp(scores).sum()
for k, p in enumerate(probs, start=1):
    print(f"Class {k}: {p:.3f}")
print("Sum of probabilities =", probs.sum().round(3))
print("Predicted class      = Class", probs.argmax() + 1)

---
# Part B — Hands-on 1 (slides 20–23)
**Dataset:** a balanced subset (7,069 records) of the *CDC Diabetes Health Indicators* dataset.
There are 21 input features (e.g. `HighBP`, `BMI`, `Age`, `PhysActivity`) and one binary target, `Diabetes_binary` (0 = no diabetes, 1 = prediabetes/diabetes).

### Step 1: Load libraries and seed the random number generator

In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Input
from tensorflow.keras.optimizers import SGD
from sklearn.model_selection import train_test_split as split
from sklearn.preprocessing import StandardScaler
from tensorflow.keras.callbacks import EarlyStopping
import pandas as pd
import tensorflow as tf
tf.keras.utils.set_random_seed(7)  # fix random seeds for reproducibility

### Step 2: Load the dataset and split it into train & test sets

In [ ]:
url = "https://raw.githubusercontent.com/wooihaw/datasets/main/cdc_diabetes_small.csv"
df = pd.read_csv(url)
print(df.shape)
df.head()

In [ ]:
X = df.drop(columns=['Diabetes_binary'])
y = df['Diabetes_binary']
print("Class balance:\n", y.value_counts())

X_train, X_test, y_train, y_test = split(X, y, test_size=0.25, random_state=7)

# Inputs must be numerical and scaled consistently (slide 13).
# Fit the scaler on the TRAINING set only, then apply it to the test set.
scaler = StandardScaler()
Xs_train = scaler.fit_transform(X_train)
Xs_test = scaler.transform(X_test)
print("Train:", Xs_train.shape, " Test:", Xs_test.shape)

### Step 3: Define the neural network model
This is a **21-12-8-1** network (slide 9 notation): 21 inputs, two hidden layers with 12 and 8 neurons, and one sigmoid output for binary classification (slide 10).

In [ ]:
# define model
model = Sequential([Input(shape=(21,))])
model.add(Dense(12, activation='relu'))
model.add(Dense(8, activation='relu'))
model.add(Dense(1, activation='sigmoid'))
model.summary()

**Check your understanding:** the first hidden layer has 21 × 12 weights + 12 biases = **264** parameters. Can you verify the other two numbers in the summary?

### Step 4: Compile the neural network model

In [ ]:
# Compile model
model.compile(loss='binary_crossentropy', optimizer=SGD(learning_rate=0.01), metrics=['accuracy'])

### Step 5: Train the neural network with early stopping
* `batch_size=8` → mini-batch training (slide 16)
* `validation_split=0.2` → 20% of the training data is held out for validation
* `EarlyStopping` stops training when `val_loss` has not improved for 10 epochs (slide 15)

In [ ]:
# Train the model
es = EarlyStopping(monitor='val_loss', mode='min', verbose=1, patience=10)
history = model.fit(Xs_train, y_train, validation_split=0.2, epochs=100, batch_size=8, callbacks=[es])

### Step 6: Evaluate the neural network model on the test set

In [ ]:
# evaluate the model
scores = model.evaluate(Xs_test, y_test)
print(f"loss: {scores[0]:.2f}")
print(f"accuracy: {scores[1]:.2%}")

### Steps 7 & 8: Plot the model accuracy and loss

In [ ]:
import matplotlib.pyplot as plt
plt.plot(history.history['accuracy'], 'b', label='train')
plt.plot(history.history['val_accuracy'], 'g', label='val')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.title('Model Accuracy')
plt.legend(loc='best')
plt.show()

plt.plot(history.history['loss'], 'b', label='train')
plt.plot(history.history['val_loss'], 'g', label='val')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('Model Loss')
plt.legend(loc='best')
plt.show()

### Making predictions (slide 19)
The sigmoid output is the **probability** of the positive class. Using a 0.5 threshold turns it into a class label.

In [ ]:
probs = model.predict(Xs_test[:10], verbose=0).ravel()
preds = (probs > 0.5).astype(int)
print(pd.DataFrame({'P(diabetes)': probs.round(3),
                    'predicted': preds,
                    'actual': y_test[:10].values}))

---
# Part C — Try it yourself
Change **one thing at a time**, re-run Steps 3–8, and record the test accuracy and the number of epochs trained.

| # | Experiment | Slide |
|---|---|---|
| 1 | Change the topology, e.g. **21-32-16-1** or **21-8-1**. Does a bigger network help on this small dataset? | 12 |
| 2 | Change the learning rate to `0.1` and then `0.001`. What happens to the loss curve? | 17 |
| 3 | Set `batch_size=1` (stochastic) and then `batch_size=len(Xs_train)` (batch). Compare speed and smoothness of the curves. | 16 |
| 4 | Replace `SGD(learning_rate=0.01)` with `'adam'` or `'rmsprop'`. | 14 |
| 5 | Add momentum: `SGD(learning_rate=0.01, momentum=0.9)`. | 18 |
| 6 | Remove the `EarlyStopping` callback and train for 100 epochs. Can you see overfitting in the plots? | 15 |
| 7 | Skip the `StandardScaler` and train on `X_train` directly. Why does performance drop? | 13 |

In [ ]:
# Use this cell for your experiments